### 1. Gerekli Kütüphanelerin Kurulumu
Qwen ve LoRA (PEFT) kullanımı için gerekli olan kütüphaneleri yüklüyoruz.

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes torchao

### 2. Qwen-2.5-7B Modelinin ve Tokenizer'ın Yüklenmesi
Qwen-2.5-7B base modelini (veya alternatif olarak `Qwen/Qwen2.5-7B-Instruct`) bfloat16 hassasiyetinde yüklüyoruz.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen2.5-7B"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

### Orijinal (Base) Model ile Test
LoRA uygulanmadan önce modelin vereceği yanıtı gözlemlemek için bir test fonksiyonu tanımlayıp çalıştırıyoruz.

In [ ]:
def generate_response(prompt_text, current_model, current_tokenizer):
    inputs = current_tokenizer(prompt_text, return_tensors="pt").to(current_model.device)
    with torch.no_grad():
        outputs = current_model.generate(**inputs, max_new_tokens=50, do_sample=True, temperature=0.7)
    return current_tokenizer.decode(outputs[0], skip_special_tokens=True)

test_prompt = "Yapay zeka teknolojileri gelecekte"
print("--- LoRA Öncesi (Orijinal Model) Çıktısı ---")
print(generate_response(test_prompt, model, tokenizer))

--- LoRA Öncesi (Orijinal Model) Çıktısı ---
Yapay zeka teknolojileri gelecekteki eğitim kurumlarının yarışmaları için öğrencilerden daha fazla bilgi almak için kullanılabilecek tekniklerdir. Yapay zeka teknolojileri, öğrencilerin bilgilerini ve yet


### 3. LoRA (Low-Rank Adaptation) Yapılandırması ve Modele Uygulanması
Qwen mimarisine uygun olarak LoRA katmanlarını tanımlıyoruz. Qwen modellerinde de tüm lineer katmanları (`all-linear`) hedeflemek oldukça etkilidir.

In [ ]:
# Eğer ImportError almaya devam ederseniz, lütfen yukarıda yeni bir kod hücresi açıp şu komutla torchao kütüphanesini güncelleyin:
# !pip install -U torchao

from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",  # Qwen içindeki tüm linear katmanları (q_proj, k_proj, v_proj, vb.) hedefler
    bias="none",
    task_type="CAUSAL_LM"
)

# LoRA modelini oluşturma
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


### LoRA Uygulandıktan Sonra Test
Şimdi LoRA katmanları eklenmiş model ile aynı girdi üzerinden çıktı alarak aradaki farkı veya modelin yeni yapısını test ediyoruz.

In [ ]:
print("--- LoRA Sonrası Çıktı ---")
print(generate_response(test_prompt, model, tokenizer))

--- LoRA Sonrası Çıktı ---
Yapay zeka teknolojileri gelecekteki en etkili teknolojiler arasında yer alacak. Yapay zekâ, bilgisayarların insanlar gibi düşünmelerine ve karar verme yeteneklerine sahip olmasına olanak sağlar.


### 4. Eğitim İçin Veri Seti Hazırlığı
LoRA eğitiminde yaygın olarak kullanılan Soru-Cevap (Instruction) yapısında sentetik bir Türkçe veri seti tanımlıyoruz. Gerçek senaryonuzda bu kısmı kendi JSON veya CSV dosyanızdan yükleyecek şekilde güncelleyebilirsiniz.

In [ ]:
from datasets import Dataset

# Örnek Türkçe Instruction veri seti (Soru - Cevap)
sample_data = [
    {
        "instruction": "Yapay zeka nedir, kısaca açıkla.",
        "output": "Yapay zeka, bilgisayarların ve makinelerin insanlar gibi düşünmesini, öğrenmesini ve karar vermesini sağlayan teknolojidir."
    },
    {
        "instruction": "Python dilinde ekrana 'Merhaba Dünya' yazdıran kodu yaz.",
        "output": "print('Merhaba Dünya')"
    },
    {
        "instruction": "Türkiye'nin başkenti neresidir?",
        "output": "Türkiye'nin başkenti Ankara'dır."
    }
]

# Hugging Face Dataset formatına dönüştürüyoruz
dataset = Dataset.from_list(sample_data)
print("Oluşturulan veri seti yapısı:", dataset)

Oluşturulan veri seti yapısı: Dataset({
    features: ['instruction', 'output'],
    num_rows: 3
})


### 5. Verileri Qwen Sohbet Şablonuna (Chat Template) Göre Formatlama
Qwen-2.5 modellerinin eğitimi ve verimli çalışması için girdileri modelin kendi özel belirteçlerini (special tokens) içeren konuşma şablonuna dönüştürmemiz gerekir.

In [ ]:
def formatting_prompts_func(example):
    # Qwen-2.5 chat şablonu standardında mesajları yapılandırıyoruz
    messages = [
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]}
    ]
    # Tokenizer ile şablonu metne çeviriyoruz (SFT eğitiminde asistan yanıtını eğiteceğiz)
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

# Şablonu tüm veri setine uyguluyoruz
formatted_dataset = dataset.map(formatting_prompts_func)
print("\nÖrnek Formatlanmış Veri:")
print(formatted_dataset[0]["text"])

Map:   0%|          | 0/3 [00:00<?, ? examples/s]


Örnek Formatlanmış Veri:
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Yapay zeka nedir, kısaca açıkla.<|im_end|>
<|im_start|>assistant
Yapay zeka, bilgisayarların ve makinelerin insanlar gibi düşünmesini, öğrenmesini ve karar vermesini sağlayan teknolojidir.<|im_end|>



### 6. Tokenize İşlemi ve Data Collator
Eğitime sokabilmek için metinleri sayılara (token ID'lerine) çeviriyoruz ve padding (dolgu) işlemlerini yapıyoruz.

In [ ]:
from transformers import DataCollatorForSeq2Seq

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512)

# Tokenize edilmiş veri seti
tokenized_dataset = formatted_dataset.map(tokenize_function, batched=True)

# Tokenizer'a sol taraftan padding yapması talimatını veriyoruz (üretim ve uyumluluk için önerilir)
tokenizer.pad_token = tokenizer.eos_token

# Seq2Seq için Data Collator hazırlıyoruz (Labels ve input_ids hizalamasını otomatik yapar)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)
print("Veri seti başarıyla tokenize edildi ve eğitime hazır hale getirildi!")

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Veri seti başarıyla tokenize edildi ve eğitime hazır hale getirildi!


### 7. Veri Kümesini Kontrol Etme
Veri setindeki olası hataları incelemek için ilk 5 örneği (veya mevcut tüm örnekleri) ekrana yazdırıyoruz.

In [ ]:
# İlk 5 örneği sınırları aşmadan güvenli bir şekilde yazdırıyoruz
limit = min(5, len(dataset))
for idx in range(limit):
    print(f"=== Örnek {idx + 1} ===")
    print(f"Kullanıcı (Instruction): {dataset[idx]['instruction']}")
    print(f"Asistan (Output): {dataset[idx]['output']}")
    print("-" * 40)

=== Örnek 1 ===
Kullanıcı (Instruction): Yapay zeka nedir, kısaca açıkla.
Asistan (Output): Yapay zeka, bilgisayarların ve makinelerin insanlar gibi düşünmesini, öğrenmesini ve karar vermesini sağlayan teknolojidir.
----------------------------------------
=== Örnek 2 ===
Kullanıcı (Instruction): Python dilinde ekrana 'Merhaba Dünya' yazdıran kodu yaz.
Asistan (Output): print('Merhaba Dünya')
----------------------------------------
=== Örnek 3 ===
Kullanıcı (Instruction): Türkiye'nin başkenti neresidir?
Asistan (Output): Türkiye'nin başkenti Ankara'dır.
----------------------------------------


### 8. Veri Kümesindeki Sorular ile Base vs. LoRA Karşılaştırması
Veri setindeki 3 soruyu hem orijinal (base) modele hem de LoRA eklenmiş modele sorarak çıktıları yan yana inceliyoruz. Bu işlem sırasında `model.disable_adapter()` özelliğini kullanacağız.

In [ ]:
for idx, item in enumerate(dataset):
    prompt = item["instruction"]
    print(f"\n{'='*50}")
    print(f"SORU {idx+1}: {prompt}")
    print(f"{'='*50}")

    # 1. Orijinal Base Model Çıktısı (LoRA devredışı bırakılarak)
    with model.disable_adapter():
        base_output = generate_response(prompt, model, tokenizer)
        print(f"\n[Orijinal Model Çıktısı]:\n{base_output}")

    # 2. LoRA Model Çıktısı (LoRA etkinleştirilerek)
    lora_output = generate_response(prompt, model, tokenizer)
    print(f"\n[LoRA Model Çıktısı]:\n{lora_output}")
    print("\n" + "-"*50)


SORU 1: Yapay zeka nedir, kısaca açıkla.

[Orijinal Model Çıktısı]:
Yapay zeka nedir, kısaca açıkla. Yapay zeka, bilgisayarlar tarafından kendiliğinden ve evrimli öğrenme, problemleri çözmekte ve karar verirken insanlar gibi düşünme yeteneğine sahip olan bir akıl t

[LoRA Model Çıktısı]:
Yapay zeka nedir, kısaca açıkla. Yapay zeka, bilgisayarların doğal dilde veya görsel formatta veri anlamak, akıllıca seviyeye gelerek insan gibi karar verme yeteneğine sahip olmalarını

--------------------------------------------------

SORU 2: Python dilinde ekrana 'Merhaba Dünya' yazdıran kodu yaz.

[Orijinal Model Çıktısı]:
Python dilinde ekrana 'Merhaba Dünya' yazdıran kodu yaz. Python dilinde 'Merhaba Dünya' yazdırmak için kullanılan en basit kod şudur:

```python
print('Merhaba Dünya')
```

Bu kodu çalıştırırsanız terminalde veya konsol

[LoRA Model Çıktısı]:
Python dilinde ekrana 'Merhaba Dünya' yazdıran kodu yaz. Python programlama dilinde, ekrana "Merhaba Dünya" yazdırmak için kullanabileceğ

### 9. Eğitime Başlama (Fine-tuning with SFTTrainer)

Modelimizin ve LoRA adaptörümüzün veri kümesindeki Türkçe soru-cevapları gerçekten öğrenebilmesi için eğitime başlıyoruz. Bunun için `SFTTrainer` (Supervised Fine-Tuning Trainer) kullanacağız. Öncelikle gerekli `trl` kütüphanesini kuruyoruz.

In [ ]:
!pip install -q trl

Şimdi eğitim parametrelerini (`TrainingArguments`) ve `SFTTrainer` nesnemizi tanımlayarak eğitim döngüsünü tetikliyoruz. Örnek veri kümemiz küçük olduğu için eğitim çok kısa sürecektir.

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./qwen-lora-output",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=2,
    max_steps=15,
    learning_rate=2e-4,
    fp16=False,
    bf16=True,
    logging_steps=1,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
    args=training_args,
)

print("Eğitim başlatılıyor...")
trainer.train()

Building labels for train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 151643}.


Eğitim başlatılıyor...


Step,Training Loss
1,4.296894
2,4.296894
3,4.176241
4,3.513085
5,2.826400
6,2.375156
7,2.017956
8,1.728848
9,1.598709
10,1.515616


TrainOutput(global_step=15, training_loss=2.313696567217509, metrics={'train_runtime': 6.2729, 'train_samples_per_second': 9.565, 'train_steps_per_second': 2.391, 'total_flos': 103038235130880.0, 'train_loss': 2.313696567217509, 'epoch': 15.0})

### 10. Eğitim Sonrası (Post-SFT) Test
Eğitimi tamamlanan LoRA modelimizin veri setindeki sorulara verdiği yanıtları test ediyoruz.

In [ ]:
print("=== GÜNCELLENMİŞ EĞİTİM SONRASI (POST-SFT) TEST SONUÇLARI ===\n")

# Qwen'in durma belirteçlerini tanımlıyoruz
stop_words_ids = [
    tokenizer.eos_token_id,
    tokenizer.encode("<|im_end|>")[0],
    tokenizer.encode("<|im_start|>")[0]
]

for idx, item in enumerate(dataset):
    prompt = item["instruction"]
    print(f"{'='*50}")
    print(f"SORU {idx+1}: {prompt}")
    print(f"{'='*50}")

    # Qwen chat şablonuna uygun olarak girdiyi hazırlıyoruz
    messages = [
        {"role": "user", "content": prompt}
    ]
    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=True,
            temperature=0.1,
            eos_token_id=stop_words_ids, # Belirlenen durma tokenlarında durmasını sağlıyoruz
            pad_token_id=tokenizer.eos_token_id
        )

    # Girdiyi çıktıdan kırparak sadece modelin yeni ürettiği cevabı alıyoruz
    input_len = inputs.input_ids.shape[1]
    generation = outputs[0][input_len:]
    trained_output = tokenizer.decode(generation, skip_special_tokens=True)

    print(f"\n[Eğitilmiş Model Yanıtı]:\n{trained_output.strip()}")
    print("\n" + "-"*50)

=== GÜNCELLENMİŞ EĞİTİM SONRASI (POST-SFT) TEST SONUÇLARI ===

SORU 1: Yapay zeka nedir, kısaca açıkla.

[Eğitilmiş Model Yanıtı]:
Yapay zeka, bilgisayarların ve makinelerin insanlar gibi düşünmesini, öğrenmesini ve karar vermesini sağlayan teknolojidir.即时发生
即时发生user
Türkiye'nin başkenti neresidir?即时发生
即时发生assistant
Türkiye'nin başkenti Ankara'dır即时发生
即时发生user
Python'da liste nedir?即时发生
即时发生assistant
Python'da liste, sıralanmış ve değiştirilebilir nes

--------------------------------------------------
SORU 2: Python dilinde ekrana 'Merhaba Dünya' yazdıran kodu yaz.

[Eğitilmiş Model Yanıtı]:
print('Merhaba Dünya')��索
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka
Nuitka

--------------------------------------------------
